# TBD1 — FreeSurfer reconstruction + BEM/scalp surfaces

**Pygers-MEG preprocessing (TBD1: step number not assigned yet)**

This notebook builds the anatomical surfaces that MEG source analysis and coregistration need.
It does three things for **one subject / one session**:

1. **`recon-all`** (FreeSurfer 8.2.0) on the subject's T1w (and T2w, if present) from the BIDS dataset.
   This reconstructs the cortical surfaces and segments the brain. It takes roughly 2–4 hours.
2. **`mne watershed_bem`** — makes the inner-skull, outer-skull and outer-skin surfaces used for the
   boundary-element model (BEM) forward model.
3. **`mne make_scalp_surfaces`** — makes high-resolution head (scalp) surfaces, used later when we
   line up the OPM sensor helmet / head scan with the MRI (coregistration).

All three steps run together as **one job on the scotty cluster** (Slurm), so you can close this notebook
while it runs. Results go to:

```
<BIDS root>/derivatives/freesurfer/sub-<label>/
```

**What you need before running this:**

- The subject's MRI already in BIDS (from `04_FIF_to_BIDS.ipynb`), with the *original, non-defaced* T1w.
- The FreeSurfer 8.2.0 container (see the 📦 box below).
- A FreeSurfer license file. FreeSurfer will not run without one. It is free: register at
  https://surfer.nmr.mgh.harvard.edu/registration.html and you will be emailed a `license.txt`.
  The license is personal and FreeSurfer does not allow sharing it, so keep it **out of your BIDS
  dataset**. Save it next to the FreeSurfer container, which is where this notebook looks by default
  (after you have created the folder in the 📦 box below).

- This notebook running in the `pygers-meg` conda environment (it uses that environment's MNE-Python).

📦 **Get the FreeSurfer singularity image once (pin a version so your reconstructions are reproducible):**

If you don't have this container, please create a folder for it and install it with these commands.

Create a hidden cache directory in your own partition, rather than the default home folder location, plus a folder for the container:

> ```bash
> export APPTAINER_CACHEDIR=/jukebox/[yourpath]/singularity-containers/cache
> mkdir -p $APPTAINER_CACHEDIR
> mkdir -p /jukebox/[yourpath]/singularity-containers/freesurfer
> ```

 
Pull the container (it is large; expect 20–40 minutes):

> ```bash
> apptainer pull \
> /jukebox/[yourpath]/singularity-containers/freesurfer/freesurfer-8.2.0.sif \
> docker://freesurfer/freesurfer:8.2.0
> ```

**How to use it:** edit the settings in the next cell, then run every cell from top to bottom.

## 1. Settings — edit these

Everything you normally need to change is in the cell below. You should not need to edit any other cell.

> This cell is tagged `parameters`, so the Pygers-MEG pipeline runner (papermill) can fill these in
> automatically from the YAML config.

In [ ]:
# ============================================================
# SETTINGS — change these for your study and subject
# ============================================================

# --- Where your data are ------------------------------------------------
# The top folder of your BIDS dataset (the folder that holds dataset_description.json
# and the sub-XX folders).
BIDS_ROOT = "/jukebox/[yourpath]/pygers-meg/data/auditory-oddball/bids"

# Subject and session labels WITHOUT the "sub-" / "ses-" prefix.
# Example: for sub-01/ses-01 write SUBJECT = "01" and SESSION = "01".
SUBJECT = "01"
SESSION = "01"

# --- FreeSurfer container and license ----------------------------------
# The folder you pulled the containers into (see the box at the top of the notebook).
CONTAINER_DIR = "/jukebox/[yourpath]/singularity-containers"
FS_SIF = f"{CONTAINER_DIR}/freesurfer/freesurfer-8.2.0.sif"   # the FreeSurfer 8.2.0 image
FS_LICENSE = f"{CONTAINER_DIR}/freesurfer/license.txt"        # your FreeSurfer license (see the intro)

# --- Which images to use -----------------------------------------------
# If True and the subject has a T2w image, recon-all uses it to refine the pial
# (outer cortex) surface (-T2 ... -T2pial). If False, only the T1w is used.
USE_T2 = True

# --- Cluster (Slurm) resources -----------------------------------------
# recon-all in FreeSurfer 8 runs on CPUs; it does not need a GPU (see the note in Step 5).
N_CPUS = 8              # CPU cores for the job
MEMORY = "48G"          # FreeSurfer 8 can use a lot of memory at its peak; 48G is a safe default
WALLTIME = "08:00:00"   # maximum run time (hours:minutes:seconds); a typical run is 2-4 hours

# Email address for Slurm to notify when the job begins, ends, or fails.
# Example: SLURM_EMAIL = "netid@princeton.edu". Leave as None for no emails.
SLURM_EMAIL = None

# --- Behaviour ----------------------------------------------------------
# If the subject's FreeSurfer folder already exists:
#   OVERWRITE = False -> a finished recon-all is kept and only the BEM/scalp steps are re-run;
#                        an unfinished/crashed one stops the notebook with an explanation.
#   OVERWRITE = True  -> the old folder is DELETED and everything is re-run from scratch.
OVERWRITE = False

# If True, this notebook waits (checking every minute) until the cluster job finishes,
# then shows the quality-control figures. If False, it submits the job and stops;
# come back later and run the cells from Step 7 onward.
WAIT_FOR_JOB = True

## 2. Set up and check the inputs

This cell loads the Python tools we need, works out all the file paths from your settings, and checks
that everything exists **before** anything is sent to the cluster. If something is missing it stops with
a message that says what to fix.

In [ ]:
# ---- Python tools ------------------------------------------------------
import json        # read/write .json files
import os          # environment variables, user name
import shutil      # find programs, delete folders
import subprocess  # run shell commands (sbatch, apptainer, ...)
import sys         # information about this Python environment
import time        # pause between job-status checks
from pathlib import Path   # convenient file paths

import mne         # MNE-Python (used here for its version number and for the QC plots)

# ---- Paths derived from the settings ---------------------------------
bids_root = Path(BIDS_ROOT)
# FreeSurfer subject name: we use the BIDS subject label (sub-01) so later MNE / MNE-BIDS-Pipeline
# steps find it automatically.
fs_subject = f"sub-{SUBJECT}"
subjects_dir = bids_root / "derivatives" / "freesurfer"   # FreeSurfer's "SUBJECTS_DIR"
fs_subject_dir = subjects_dir / fs_subject                 # this subject's FreeSurfer folder
# The job script and its log files are kept with the derivatives so you can see exactly what was run.
slurm_dir = subjects_dir / "code" / "slurm"
anat_dir = bids_root / f"sub-{SUBJECT}" / f"ses-{SESSION}" / "anat"

# The 'mne' command-line program from THIS conda environment (the one running the notebook).
# We run it inside the FreeSurfer container so it can call FreeSurfer's tools.
env_prefix = Path(sys.prefix)
mne_cli = env_prefix / "bin" / "mne"

problems = []   # we collect every problem first, then report them all at once

# Did the user replace the placeholder BIDS path?
if "[yourpath]" in BIDS_ROOT or "[yourpath]" in CONTAINER_DIR:
    problems.append("BIDS_ROOT or CONTAINER_DIR still contains the [yourpath] placeholder — set them in Step 1.")
elif not (bids_root / "dataset_description.json").is_file():
    problems.append(f"No dataset_description.json in BIDS_ROOT ({bids_root}). Is this the BIDS root?")

# Find the T1w (and T2w). We skip any defaced copies, because FreeSurfer and coregistration
# need the original face/scalp.
def find_image(suffix):
    # Look for e.g. sub-01_ses-01_..._T1w.nii.gz in the anat folder
    hits = sorted(p for p in anat_dir.glob(f"sub-{SUBJECT}_ses-{SESSION}*_{suffix}.nii*")
                  if "defaced" not in p.name)
    return hits

t1_hits = find_image("T1w")
t2_hits = find_image("T2w") if USE_T2 else []

if not anat_dir.is_dir():
    problems.append(f"No anat folder found at {anat_dir}")
elif len(t1_hits) == 0:
    problems.append(f"No T1w image found in {anat_dir}")
elif len(t1_hits) > 1:
    problems.append("More than one T1w found — keep only the one you want FreeSurfer to use:\n    "
                    + "\n    ".join(str(p) for p in t1_hits))
if len(t2_hits) > 1:
    problems.append("More than one T2w found — keep only one, or set USE_T2 = False:\n    "
                    + "\n    ".join(str(p) for p in t2_hits))

t1_path = t1_hits[0] if len(t1_hits) == 1 else None
t2_path = t2_hits[0] if len(t2_hits) == 1 else None

# FreeSurfer container and license (install instructions are in the box at the top of the notebook)
if not Path(FS_SIF).is_file():
    problems.append(f"FreeSurfer container not found at {FS_SIF}\n"
                    "    -> pull it with the commands in the box at the top of this notebook.")
if not Path(FS_LICENSE).is_file():
    problems.append(f"FreeSurfer license not found at {FS_LICENSE}\n"
                    "    -> see the intro of this notebook for how to get one and where to save it.")

# The mne program in this environment
if not mne_cli.is_file():
    problems.append(f"Could not find the 'mne' program at {mne_cli}. "
                    "Run this notebook with the pygers-meg conda environment's kernel.")

# Container program: newer clusters call it 'apptainer', older ones 'singularity'
container_cmd = shutil.which("apptainer") or shutil.which("singularity")
if container_cmd is None:
    problems.append("Neither 'apptainer' nor 'singularity' was found on this machine.")

# Slurm
if shutil.which("sbatch") is None:
    problems.append("'sbatch' not found — run this notebook on the scotty cluster.")

# ---- Report --------------------------------------------------------------
print(f"BIDS root        : {bids_root}")
print(f"Subject/session  : sub-{SUBJECT} / ses-{SESSION}")
print(f"T1w              : {t1_path}")
print(f"T2w              : {t2_path if t2_path else '(none — T1w only)'}")
print(f"Output folder    : {fs_subject_dir}")
print(f"MNE-Python       : {mne.__version__}  ({mne_cli})")
print(f"FreeSurfer image : {FS_SIF}")
print(f"Container program: {container_cmd}")

if problems:
    raise RuntimeError("Please fix the following before continuing:\n  - " + "\n  - ".join(problems))
print("\nAll inputs found.")

## 3. Quick test of the container

Before we tie up a cluster node for hours, we check (in a few seconds) that:

- FreeSurfer inside the container starts and reports version 8.2.0, and
- MNE-Python from your conda environment can run **inside** the container (it needs FreeSurfer's tools
  for the BEM and scalp steps).

`--cleanenv` stops settings from your own login (like your conda `PATH`) leaking into the container;
`--bind` makes our data folders visible inside it.

In [ ]:
# Folders the container needs to see. Apptainer mounts each one at the same path inside the container.
# Some paths are really links to another place. For example ~/miniconda3 may be a link to
# /jukebox/<lab>/miniconda3. Apptainer already shows your home folder inside the container, links
# included, so we mount the REAL folder (found with os.path.realpath) at its real path; the link in
# your home folder then points to it, and both paths work inside the container.
def real(p):
    # The real location of a file or folder, with all links followed. Inside the container we
    # always use these real paths, because the links (e.g. /jukebox/..., ~/miniconda3) may not exist there.
    return os.path.realpath(p)

mount_sources = [bids_root, env_prefix, Path(FS_LICENSE).parent, Path(t1_path).parent]
if t2_path is not None:
    mount_sources.append(Path(t2_path).parent)
bind_paths = sorted({real(p) for p in mount_sources})
print("Folders mounted in the container:")
for b in bind_paths:
    print("   ", b)

# The start of every command we run inside the container
container_prefix = [
    container_cmd, "exec", "--cleanenv",
    *[arg for p in bind_paths for arg in ("--bind", p)],
    "--env", f"FS_LICENSE={real(FS_LICENSE)}",
    "--env", f"SUBJECTS_DIR={real(subjects_dir)}",
    "--env", "FS_ALLOW_DEEP=1",            # lets FreeSurfer 8 use its deep-learning steps
    str(FS_SIF),
]

def run_in_container(*args):
    # Run one command inside the container and return what it printed
    result = subprocess.run(container_prefix + list(args), capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()

rc, out = run_in_container("recon-all", "--version")
print("recon-all --version:", out.splitlines()[-1] if out else "(no output)")
if rc != 0 or "8.2.0" not in out:
    raise RuntimeError("FreeSurfer 8.2.0 did not start correctly inside the container:\n" + out)

# How we start MNE inside the container: the environment's own Python, at its REAL location on
# /jukebox, with "-m mne". This avoids any path that goes through a link (like ~/miniconda3).
mne_cmd = [os.path.join(real(env_prefix), "bin", "python"), "-m", "mne"]

rc, out = run_in_container(*mne_cmd, "--version")
print("mne --version      :", out.splitlines()[-1] if out else "(no output)")
if rc != 0:
    raise RuntimeError("MNE-Python could not run inside the container.\nCommand: "
                       + " ".join(container_prefix + mne_cmd + ["--version"]) + "\n" + out)

print("\nContainer test passed.")

## 4. Check for an existing FreeSurfer run

FreeSurfer writes `scripts/recon-all.done` when it finishes successfully. We use that file to decide
what to do (see the `OVERWRITE` setting in Step 1).

In [ ]:
done_file = fs_subject_dir / "scripts" / "recon-all.done"
error_file = fs_subject_dir / "scripts" / "recon-all.error"

# Worked out automatically below from OVERWRITE and what is already on disk; do not edit.
# True = the job runs recon-all; False = it only redoes the BEM/scalp steps.
_run_recon = True

if fs_subject_dir.exists():
    if OVERWRITE:
        print(f"OVERWRITE = True: deleting the old folder {fs_subject_dir}")
        shutil.rmtree(fs_subject_dir)
    elif done_file.is_file():
        print("recon-all already finished for this subject — it will NOT be re-run.")
        print("The job will only (re)make the BEM and scalp surfaces.")
        _run_recon = False
    else:
        # recon-all refuses to start if the folder exists, and a half-finished run is not usable
        reason = "it crashed (recon-all.error exists)" if error_file.is_file() else "it never finished"
        raise RuntimeError(
            f"{fs_subject_dir} exists but {reason}.\n"
            f"Check {fs_subject_dir / 'scripts' / 'recon-all.log'} to see why, then set "
            "OVERWRITE = True in Step 1 to start again.")
else:
    print("No previous FreeSurfer run — recon-all will run from the start.")

# Create the output folders
slurm_dir.mkdir(parents=True, exist_ok=True)

## 5. Write the cluster job script

The cell below writes a Slurm job script (a small bash file) and shows it to you. The job runs:

```
recon-all -all -i T1w [-T2 T2w -T2pial] -parallel -openmp N
mne watershed_bem --atlas --gcaatlas
mne make_scalp_surfaces --force
```

We also record the dataset in `derivatives/freesurfer/dataset_description.json` (required by BIDS for a
derivatives folder).

In [ ]:
# ---- BIDS derivatives description (written once) ------------------------------
desc_file = subjects_dir / "dataset_description.json"
if not desc_file.is_file():
    # Use the same BIDS version as the main dataset
    raw_desc = json.loads((bids_root / "dataset_description.json").read_text())
    description = {
        "Name": "FreeSurfer reconstructions with MNE BEM and scalp surfaces",
        # Copied from the main dataset so both match (1.11.1 is the Pygers-MEG default if it is missing)
        "BIDSVersion": raw_desc.get("BIDSVersion", "1.11.1"),
        "DatasetType": "derivative",
        "GeneratedBy": [
            {"Name": "FreeSurfer", "Version": "8.2.0",
             "Container": {"Type": "apptainer", "URI": "docker://freesurfer/freesurfer:8.2.0"}},
            {"Name": "MNE-Python", "Version": mne.__version__},
        ],
    }
    desc_file.write_text(json.dumps(description, indent=4) + "\n")
    print(f"Wrote {desc_file}")

# ---- Pieces of the job script ----------------------------------------------------
# The recon-all command (only used when _run_recon is True)
# (real(...) = the path as it appears inside the container; see Step 3)
recon_cmd = f'recon-all -all -s {fs_subject} -sd "{real(subjects_dir)}" -i "{real(t1_path)}"'
if t2_path is not None:
    recon_cmd += f' -T2 "{real(t2_path)}" -T2pial'
recon_cmd += f" -parallel -openmp {N_CPUS}"

# The container part, written once as a bash "array" so every step uses identical settings.
# shlex.quote adds safe quoting in case a path contains spaces.
import shlex
container_args = [Path(container_cmd).name, "exec", "--cleanenv",
                  *[a for p in bind_paths for a in ("--bind", p)],
                  "--env", f"FS_LICENSE={real(FS_LICENSE)}", "--env", f"SUBJECTS_DIR={real(subjects_dir)}",
                  "--env", "FS_ALLOW_DEEP=1", "--env", f"OMP_NUM_THREADS={N_CPUS}", str(FS_SIF)]
container_line = " ".join(shlex.quote(a) for a in container_args)

job_name = f"fs_{fs_subject}"
sbatch_lines = [
    "#!/bin/bash",
    f"#SBATCH --job-name={job_name}",
    "#SBATCH --nodes=1",
    "#SBATCH --ntasks=1",
    f"#SBATCH --cpus-per-task={N_CPUS}",
    f"#SBATCH --mem={MEMORY}",
    f"#SBATCH --time={WALLTIME}",
    f"#SBATCH --output={slurm_dir}/%x_%j.log",   # %x = job name, %j = job number
]
# Optional email notifications when the job begins, ends, or fails
if SLURM_EMAIL:
    sbatch_lines.append(f"#SBATCH --mail-user={SLURM_EMAIL}")
    sbatch_lines.append("#SBATCH --mail-type=BEGIN,END,FAIL")


job_script = "\n".join(sbatch_lines) + f'''

# Stop at the first error, and treat unset variables as errors
set -euo pipefail

# Every FreeSurfer/MNE command below runs inside the FreeSurfer 8.2.0 container
FS=({container_line})

echo "Job $SLURM_JOB_ID on $(hostname), started $(date)"
"${{FS[@]}}" recon-all --version

# ---- 1. recon-all ---------------------------------------------------------------
if [ "{int(_run_recon)}" = "1" ]; then
    echo "=== recon-all ($(date)) ==="
    "${{FS[@]}}" {recon_cmd}
else
    echo "=== recon-all already done; skipping ==="
fi

# ---- 2. BEM surfaces (inner skull, outer skull, outer skin) -------------------------
echo "=== mne watershed_bem ($(date)) ==="
"${{FS[@]}}" {shlex.join(mne_cmd)} watershed_bem --subject={fs_subject} --subjects-dir="{real(subjects_dir)}" \\
    --overwrite --atlas --gcaatlas --verbose

# ---- 3. High-resolution scalp surfaces (for coregistration) -----------------------
echo "=== mne make_scalp_surfaces ($(date)) ==="
"${{FS[@]}}" {shlex.join(mne_cmd)} make_scalp_surfaces --subject={fs_subject} --subjects-dir="{real(subjects_dir)}" \\
    --overwrite --force --verbose

echo "All steps finished $(date)"
'''

job_file = slurm_dir / f"{job_name}.sh"
job_file.write_text(job_script)
print(f"Job script written to {job_file}\n")
print(job_script)

## 6. Submit the job to the cluster

`sbatch` puts the job in the queue and gives back a job number. The job's log will be in
`derivatives/freesurfer/code/slurm/` (the file name ends with the job number).

In [ ]:
# --parsable makes sbatch print only the job number
result = subprocess.run(["sbatch", "--parsable", str(job_file)], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("sbatch failed:\n" + result.stderr)

job_id = result.stdout.strip().split(";")[0]
log_file = slurm_dir / f"{job_name}_{job_id}.log"
print(f"Submitted job {job_id}")
print(f"Log file: {log_file}")
print(f"\nYou can check it yourself in a terminal with:  squeue -j {job_id}")

## 7. Wait for the job and check its status

If `WAIT_FOR_JOB = True`, this cell checks the job every minute and prints its state until it ends.
It then shows the last lines of the log.

If you closed the notebook while the job was running: re-run **Steps 1–2** (settings and checks), set
`job_id` by hand in the first line of the cell below (e.g. `job_id = "1234567"`), and run from here.

In [ ]:
# job_id = "1234567"   # <- only needed if you came back later; type your job number here

def job_state(jid):
    # Ask Slurm for the job's state (PENDING, RUNNING, COMPLETED, FAILED, TIMEOUT, ...)
    out = subprocess.run(["sacct", "-j", str(jid), "-X", "-n", "-o", "State"],
                         capture_output=True, text=True).stdout.strip()
    return out.split()[0] if out else "UNKNOWN"

finished_states = {"COMPLETED", "FAILED", "CANCELLED", "TIMEOUT", "OUT_OF_MEMORY", "NODE_FAIL"}

state = job_state(job_id)
if WAIT_FOR_JOB:
    last = None
    while state not in finished_states:
        if state != last:   # only print when the state changes
            print(f"{time.strftime('%H:%M')}  job {job_id}: {state}")
            last = state
        time.sleep(60)      # wait one minute between checks
        state = job_state(job_id)

print(f"{time.strftime('%H:%M')}  job {job_id}: {state}")

# Show the end of the job log
log_file = slurm_dir / f"{job_name}_{job_id}.log"
if log_file.is_file():
    print("\n--- last 30 lines of the log ---")
    print("\n".join(log_file.read_text().splitlines()[-30:]))

if state != "COMPLETED" and state in finished_states:
    raise RuntimeError(f"The job ended with state {state}. Read the log above / {log_file}. "
                       f"FreeSurfer's own log is {fs_subject_dir / 'scripts' / 'recon-all.log'}.")

## 8. Quality control

First we check that every expected output file exists. Then we draw the BEM surfaces on top of the MRI:
the **inner skull** (yellow), **outer skull** and **outer skin** should follow the anatomy smoothly, not
cut into the brain or wander outside the head. The red/blue lines are FreeSurfer's white-matter surfaces.

Look carefully around the eyes, the base of the skull and the top of the head — those are where
problems usually show up.

In [ ]:
# Files the later preprocessing steps rely on
bem_dir = fs_subject_dir / "bem"
expected = [
    fs_subject_dir / "scripts" / "recon-all.done",   # recon-all finished
    fs_subject_dir / "surf" / "lh.white",
    fs_subject_dir / "surf" / "rh.white",
    fs_subject_dir / "surf" / "lh.pial",
    fs_subject_dir / "surf" / "rh.pial",
    bem_dir / "inner_skull.surf",                     # from watershed_bem
    bem_dir / "outer_skull.surf",
    bem_dir / "outer_skin.surf",
    bem_dir / f"{fs_subject}-head.fif",               # from make_scalp_surfaces
    bem_dir / f"{fs_subject}-head-dense.fif",
]
missing = [p for p in expected if not p.exists()]
for p in expected:
    print(("OK      " if p.exists() else "MISSING ") + str(p.relative_to(subjects_dir)))
if missing:
    raise RuntimeError("Some outputs are missing — check the job log.")

In [ ]:
# Coronal slices through the head with the BEM and white-matter surfaces drawn on top
import matplotlib.pyplot as plt    # plotting (used here only to close the figure; see below)

fig = mne.viz.plot_bem(subject=fs_subject, subjects_dir=subjects_dir,
                       orientation="coronal", brain_surfaces="white",
                       slices=list(range(60, 200, 20)), show=False)
plt.close(fig)   # the last line below still shows it; closing stops it from appearing again later
fig

In [ ]:
# The same check from the side (sagittal slices)
fig = mne.viz.plot_bem(subject=fs_subject, subjects_dir=subjects_dir,
                       orientation="sagittal", brain_surfaces="white",
                       slices=list(range(70, 190, 20)), show=False)
plt.close(fig)   # the last line below still shows it; closing stops it from appearing again later
fig

### 3D views of the scalp surface

This is the head surface the coregistration notebook (TBD2) lines up with the participant's head-shape scan
(coregistration), so it should look like a complete, smooth head. Check that:

- the whole face and nose are there, not cut off by the edge of the MRI scan
- there are no holes, spikes or flat "shelves"
- nothing extra is stuck to the head (for example padding or headphones from the scanner)

The pictures show the participant's face, so don't share this notebook's output outside the lab.

In [ ]:
# ---- 3D views of the scalp (head) surface -------------------------------------------
import numpy as np                 # number arrays
import matplotlib.pyplot as plt    # plotting
from matplotlib.colors import LightSource   # where the light comes from

plt.close("all")   # close any figures left open by earlier cells, so only the views below are shown

bem_dir = fs_subject_dir / "bem"

# The "medium" scalp surface (about 30,000 triangles). It looks the same as the dense one at
# this size but draws much faster.
# Shrinking the surface to this size usually leaves a few tiny flaws in how the triangles connect
# ("topological defects"). They don't matter for a picture, so on_defects="warn" prints a warning
# instead of stopping. (This is also why make_scalp_surfaces needed --force.)
head_file = bem_dir / f"{fs_subject}-head-medium.fif"
head = mne.read_bem_surfaces(head_file, on_defects="warn", verbose=False)[0]   # the file holds one surface
verts = head["rr"] * 1000    # corner points (vertices) of the triangles, metres -> millimetres
tris = head["tris"]          # each row lists the 3 vertices that make one triangle

# Six viewing directions: (title, elevation, azimuth) in degrees.
# In FreeSurfer coordinates x points to the right ear, y to the nose and z to the top of the head.
# Elevation tilts the camera up from the horizontal; azimuth turns it around the head.
views = [
    ("Front",        0,   90),
    ("Left",         0,  180),
    ("Right",        0,    0),
    ("Back",         0,  -90),
    ("Top",         90,  -90),
    ("Front-left",  20,  135),
]

fig = plt.figure(figsize=(15, 9))
for k, (title, elev, azim) in enumerate(views, start=1):
    ax = fig.add_subplot(2, 3, k, projection="3d")
    # Put the light just above and to the side of the camera, so every view is evenly lit
    light = LightSource(azdeg=90 - azim + 25, altdeg=min(elev + 35, 90))
    # Draw the surface as shaded triangles in a skin colour
    ax.plot_trisurf(verts[:, 0], verts[:, 1], verts[:, 2], triangles=tris,
                    color="#e0b9a0", linewidth=0, antialiased=False, shade=True,
                    lightsource=light)
    ax.view_init(elev=elev, azim=azim)                    # point the camera
    ax.set_box_aspect(np.ptp(verts, axis=0), zoom=1.2)    # equal scale on all axes, so no stretching
    ax.set_axis_off()                                     # hide the axes and grid
    ax.set_title(title, fontsize=14)

fig.suptitle(f"{fs_subject}: scalp surface ({len(verts):,} points, {len(tris):,} triangles)",
             fontsize=16)
plt.tight_layout()
plt.show()

## 9. Summary of this subject

This cell writes a short **plain account of what this notebook did for this subject**: the input images,
the FreeSurfer run, a few of FreeSurfer's own numbers, the BEM and scalp surfaces, and the cluster job.
It reads everything from the files the run left behind (for example FreeSurfer's `scripts/recon-all.done`
and `stats/aseg.stats`), so it is correct even if you come back later and only run Steps 1–2 first.

It is a record for you and your lab (for example, to look back on or to compare subjects), not text for
a paper. A methods description for the whole preprocessing will come in the last preprocessing notebook.

The summary is shown below and saved as `sub-<label>_desc-summary.md` in the subject's FreeSurfer folder,
`derivatives/freesurfer/sub-<label>/`. Run it after the job has finished.

In [ ]:
# ---- Python tools for the summary -------------------------------------------------------------
import re                                         # find numbers inside log text
import warnings                                   # catch MNE's warning about surface defects
from datetime import datetime                     # today's date
import nibabel as nib                             # reads MRI image headers (voxel sizes)
from IPython.display import Markdown, display     # shows formatted text in the notebook


def fmt_n(x):
    # A number with thousands separators, e.g. 12345 -> "12,345"
    return f"{int(x):,}"


def fmt_vox(zooms):
    # Voxel size as text: "0.8 mm" if all three sides are equal, else "0.8 × 0.8 × 1.2 mm"
    z = [round(float(v), 2) for v in zooms[:3]]
    return f"{z[0]:g} mm" if len(set(z)) == 1 else " × ".join(f"{v:g}" for v in z) + " mm"


def aseg_measure(name):
    # One number from FreeSurfer's stats/aseg.stats, e.g. "eTIV" or "lhSurfaceHoles".
    # Those lines look like: "# Measure ..., lhSurfaceHoles, <description>, 10, unitless"
    for line in (fs_subject_dir / "stats" / "aseg.stats").read_text().splitlines():
        if line.startswith("# Measure") and f", {name}," in line:
            return float(line.split(",")[-2])
    return None


who = f"sub-{SUBJECT}"
bem_dir = fs_subject_dir / "bem"
parts = []

# ---- What recon-all did: read from the record FreeSurfer writes when it finishes ----------------
# scripts/recon-all.done has one "KEY value" line each for the version, times, computer and command.
recon_info = {}
for line in (fs_subject_dir / "scripts" / "recon-all.done").read_text().splitlines():
    key, _, value = line.partition(" ")
    if key.isupper():
        recon_info[key] = value.strip()
recon_args = recon_info.get("CMDARGS", "").split()      # the exact recon-all command options


def recon_option(flag):
    # The value given after an option in the recon-all command, e.g. "-i" -> the T1w path
    return recon_args[recon_args.index(flag) + 1] if flag in recon_args else None


t1_used = recon_option("-i")
t2_used = recon_option("-T2")
fs_version = recon_info.get("VERSION", "unknown").split()[0]   # e.g. "8.2.0"

# ---- Inputs -------------------------------------------------------------------------------------------
try:
    t1_vox = nib.load(t1_used).header.get_zooms()
    orig_vox = nib.load(fs_subject_dir / "mri" / "orig.mgz").header.get_zooms()
    resampled = max(abs(a - b) for a, b in zip(t1_vox[:3], orig_vox[:3])) > 0.01
    vox_text = (f" ({fmt_vox(t1_vox)} voxels; FreeSurfer resampled it to its standard {fmt_vox(orig_vox)} "
                f"voxels before processing)" if resampled else f" ({fmt_vox(t1_vox)} voxels)")
except Exception:          # the image could not be read (e.g. moved since); leave the voxel size out
    vox_text = ""
inputs = f"The T1w image was `{Path(t1_used).name}`{vox_text}"
inputs += (f", and the T2w image `{Path(t2_used).name}` was used to refine the outer (pial) surface "
           "of the cortex." if t2_used else ". No T2w image was used.")
parts.append(f"**Inputs.** {inputs}")

# ---- Reconstruction ------------------------------------------------------------------------------
recon = (f"FreeSurfer {fs_version} `recon-all -all` made the FreeSurfer subject `{fs_subject}`, running in "
         f"the FreeSurfer container on the cluster node {recon_info.get('HOST', '?')}"
         + (f" with {recon_option('-openmp')} threads" if recon_option("-openmp") else "")
         + f". It started {recon_info.get('START_TIME', '?')} and took "
         f"{float(recon_info.get('RUNTIME_HOURS', 'nan')):.1f} hours.")
if globals().get("_run_recon") is False:   # Step 4 found a finished run and kept it
    recon += " This time recon-all was not run again; only the BEM and scalp surfaces were remade."
n_white = [len(mne.read_surface(fs_subject_dir / "surf" / f"{h}.white")[0]) for h in ("lh", "rh")]
recon += (f" The cortical surfaces have {fmt_n(n_white[0])} points on the left and {fmt_n(n_white[1])} "
          "on the right.")
parts.append(f"**Reconstruction.** {recon}")

# ---- FreeSurfer's own numbers worth a look ----------------------------------------------------------
holes = [aseg_measure("lhSurfaceHoles"), aseg_measure("rhSurfaceHoles")]
etiv, stiv = aseg_measure("eTIV"), aseg_measure("sTIV")
checks = []
if None not in holes:
    checks.append(f"Before FreeSurfer fixed them, the cortical surfaces had {holes[0]:.0f} holes on the left "
                  f"and {holes[1]:.0f} on the right (fewer usually means a cleaner scan).")
if etiv:
    checks.append(f"The estimated intracranial volume is {etiv / 1e6:.2f} L"
                  + (f" ({stiv / 1e6:.2f} L from the SynthSeg segmentation)" if stiv else "") + ".")
if checks:
    parts.append("**FreeSurfer numbers.** " + " ".join(checks))

# ---- BEM surfaces -------------------------------------------------------------------------------------
bem_tris = {name: len(mne.read_surface(bem_dir / f"{name}.surf")[1])
            for name in ("inner_skull", "outer_skull", "outer_skin")}
bem_date = datetime.fromtimestamp((bem_dir / "inner_skull.surf").stat().st_mtime)
same = len(set(bem_tris.values())) == 1
parts.append(
    "**BEM surfaces.** `mne watershed_bem` (with FreeSurfer's atlas options) made the inner skull, outer "
    "skull and outer skin surfaces"
    + (f" ({fmt_n(bem_tris['inner_skull'])} triangles each)" if same else
       " (" + ", ".join(f"{k.replace('_', ' ')} {fmt_n(v)}" for k, v in bem_tris.items()) + " triangles)")
    + f" on {bem_date:%Y-%m-%d}.")

# ---- Scalp surfaces ---------------------------------------------------------------------------------------
scalp = {}
for kind in ("dense", "medium", "sparse"):
    with warnings.catch_warnings(record=True) as caught:   # keep MNE's defect warning, don't print it
        warnings.simplefilter("always")
        surf = mne.read_bem_surfaces(bem_dir / f"{fs_subject}-head-{kind}.fif", on_defects="warn",
                                     verbose=False)[0]
    found = [re.search(r"(\d+) / \d+ vertices have fewer than three", str(w.message)) for w in caught]
    scalp[kind] = (len(surf["tris"]), sum(int(m.group(1)) for m in found if m))
scalp_text = (f"`mne make_scalp_surfaces` made the scalp surface in three sizes: dense "
              f"({fmt_n(scalp['dense'][0])} triangles, used for coregistration), medium "
              f"({fmt_n(scalp['medium'][0])}) and sparse ({fmt_n(scalp['sparse'][0])}).")
flawed = [f"{kind} ({fmt_n(n)} points)" for kind, (_, n) in scalp.items() if n]
if flawed:
    scalp_text += (" Shrinking the surface left a few points with incomplete connections in the "
                   + " and ".join(flawed) + " version; that doesn't matter for pictures.")
parts.append(f"**Scalp surfaces.** {scalp_text}")

# ---- The cluster job -----------------------------------------------------------------------------------
# The newest job log that reached the end is the job that made the files above.
logs = sorted(slurm_dir.glob(f"fs_{fs_subject}_*.log"), key=lambda p: p.stat().st_mtime)
finished = [p for p in logs if "All steps finished" in p.read_text(errors="ignore")]
if finished:
    jid = finished[-1].stem.split("_")[-1]
    job = f"The work ran as Slurm job {jid}"
    # Ask Slurm how long it took and how much memory it used (skipped quietly if it can't tell us)
    try:
        acct = subprocess.run(["sacct", "-j", jid, "-n", "-P", "-o", "JobID,Elapsed,ReqMem,MaxRSS"],
                              capture_output=True, text=True).stdout.splitlines()
    except OSError:        # sacct isn't available here
        acct = []
    rows = [r.split("|") for r in acct if r.count("|") == 3]
    if rows:
        def to_gb(text):
            # Slurm memory like "22541312K", "48G" or "48000M" -> gigabytes (None if blank)
            m = re.match(r"([\d.]+)([KMGT]?)", text)
            return float(m.group(1)) * {"K": 1e-6, "M": 1e-3, "G": 1, "T": 1e3, "": 1e-9}[m.group(2)] if m else None
        peak = max((to_gb(r[3]) or 0 for r in rows), default=0)
        asked = to_gb(rows[0][2])
        job += f", which took {rows[0][1]}"
        if peak:
            job += f" and used at most {peak:.0f} GB of memory" + (f" ({asked:.0f} GB requested)" if asked else "")
    failed = len(logs) - len(finished)
    job += (f", after {failed} earlier attempt{'s' if failed > 1 else ''} that failed." if failed else ".")
    parts.append(f"**Cluster job.** {job}")

# ---- Outputs and software -----------------------------------------------------------------------------------
parts.append(
    f"**Saved.** The reconstruction, BEM and scalp surfaces are in "
    f"`{fs_subject_dir.relative_to(bids_root)}`; the job script and its logs are in "
    f"`{slurm_dir.relative_to(bids_root)}`. Run with FreeSurfer {fs_version} (container "
    f"`{Path(FS_SIF).name}`) and MNE-Python {mne.__version__}; summary written {datetime.now():%Y-%m-%d}.")

summary_md = "\n\n".join([f"### Summary: {who} (FreeSurfer)", *parts])
summary_file = fs_subject_dir / f"{fs_subject}_desc-summary.md"
summary_file.write_text(summary_md + "\n")
display(Markdown(summary_md))
print("Saved:", summary_file.relative_to(bids_root))

## Done

`derivatives/freesurfer/sub-<label>/` now holds the FreeSurfer reconstruction, the BEM surfaces, the
scalp surfaces and the summary of this subject (`sub-<label>_desc-summary.md`, Step 9). The coregistration notebook (TBD2) uses the scalp surface to coregister the MEG sensors to this MRI.